# 📊 Billing Summary Report Generator — Production

Reads the **Base File** (targets) and the **Billing Report** (achievement) from `input_files/`
and writes a timestamped report to `output_files/`, styled to match the reference
`Output.xlsx` exactly (light-blue summary band, navy per-person header block, medium outer
borders, `—` for missing targets).

This report goes to the CTO/CXO, so the notebook is written to fail loudly rather than
silently produce a wrong number: every stage prints where its numbers came from, and Step 8
cross-checks totals before the file is saved.

**Only one number in this notebook is not read from a file** — the Mouser targets
(Step 1/6), because the Base File has no Mouser rows at all.

### Permanent data-quality fixes baked in (apply every month, not just this one)
1. **Achievement = "Basic Value"**, not "Total Value" — the Billing Report's own grand-total
   row (Line 1 of the Line/Mouser sheets) confirms Basic Value is the pre-tax figure that adds
   up to the numbers actually reported.
2. **JNS INSTRUMENTS LIMITED** sits inside the USD sheet like an ordinary customer but is
   tracked as its own **"JNS-TFT"** business line. Carved out automatically
   (toggle: `INCLUDE_JNS_TFT`).
3. **Salesperson names are matched by similarity, not exact string**, so real spelling
   variants ("DHIRENDRA" vs "DHIRENDER") don't silently split one person's numbers in two.
4. **The report only lists the salespeople in `REPORT_PERSONS`** (Step 1) — this is the one
   deliberately hardcoded list in the notebook, because who goes in a CTO/CXO report is a
   business decision, not something to infer from whoever happened to bill this month.

**Run order:** Restart & Run All.

In [3]:
import re
import colorsys
import unicodedata
from datetime import datetime
from difflib import SequenceMatcher
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

RUN_STARTED = datetime.now()
print(f'✓ Libraries loaded — run started {RUN_STARTED:%Y-%m-%d %H:%M:%S}')

✓ Libraries loaded — run started 2026-08-23 07:42:51


### Step 1: Settings

Everything you might need to touch for a future month lives here. Nothing below this cell should need editing in normal operation.

In [4]:
INPUT_FOLDER  = 'input_files'      # source files: the Base File + the Billing Report
OUTPUT_FOLDER = 'output_files'     # every generated report lands here, never overwritten

HEADER_ROW = 1     # 0 = headers on Excel row 1, 1 = headers on row 2 (both source files have
                    # a totals row on row 1 and real headers on row 2)

# Reporting units
CR   = 10_000_000     # INR Crore
LAKH =    100_000     # INR Lakh
MILL =  1_000_000     # Millions USD

# ── The report roster (hardcoded on purpose) ────────────────────────────────
# This is the CTO/CXO-facing list: which salespeople appear, in which order, and under what
# exact display spelling. The Base File / Billing Report will always contain more names than
# this (new joiners, small accounts, data-entry variants) — this list is the deliberate
# business decision of who this particular report is about. Keyed by the CANONICAL name
# (after fuzzy name-matching in Step 3b) -> the exact display spelling to print.
REPORT_PERSONS = [
    ('KULDEEP KUMAR',        'Kuldeep Kumar'),
    ('ARPIT SINGH',          'Arpit Singh'),
    ('MAHESH KUMAR MITTAL',  'Mahesh Kumar'),
    ('HARSH DOSHI',          'Harsh Doshi'),
    ('SURAJ PANDEY',         'Suraj Pandey'),
    ('ABHISHEKH CHOUDHARY',  'Abhishekh Chaudhry'),
]

# ── The only hand-entered numbers in this notebook ──────────────────────────
# The Base File has NO Mouser rows at all, so Mouser targets cannot be computed — they must
# be entered here directly. Confirmed with Ankur (Aug'26): Harsh Doshi = 55, Abhishekh = 3
# Lakhs. Everyone else defaults to the flat guess below until a real number is confirmed.
MOUSER_TARGET_LAKH_DEFAULT = 13
MOUSER_TARGET_LAKH_OVERRIDES = {
    'HARSH DOSHI': 55,
    'ABHISHEKH CHOUDHARY': 3,
    # add more 'NAME IN CAPS': lakhs  as real numbers get confirmed
}

# ── JNS (TFT) carve-out ─────────────────────────────────────────────────────
# JNS INSTRUMENTS LIMITED shows up inside the USD sheet like any other USD-SG customer, but
# is booked/tracked as its own "JNS (TFT)" line in practice. Set to False to fold it back
# into plain USD-SG if this stops being a distinct business line in future.
INCLUDE_JNS_TFT    = True
JNS_CUSTOMER_MATCH = 'JNS'          # matched against Customer Name, case-insensitive

# ── Salesperson name matching ───────────────────────────────────────────────
# Two raw names are treated as the same person when their similarity is >= this threshold
# (1.0 = identical). 0.90 catches things like "DHIRENDRA PANDEY" / "DHIRENDER PANDEY" without
# merging genuinely different people — every merge made is printed in Step 3b for a human to
# check.
NAME_MATCH_THRESHOLD = 0.90

# Escape hatch: force two specific raw names together only if the automatic matching above
# ever gets a real pair wrong. Empty by default — keep it that way unless Step 3b's printed
# merge list shows a mistake.
MANUAL_NAME_OVERRIDES = {
    # 'RAW NAME AS SEEN IN A FILE': 'CANONICAL NAME TO REPORT UNDER',
}

# The Base File's month columns are named like "Aug_FY2627_Val" — note July/Sept are spelled
# out in full (not "Jul"/"Sep"), unlike pandas' usual 3-letter month abbreviations.
MONTH_PREFIX = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun', 7: 'July',
                8: 'Aug', 9: 'Sept', 10: 'Oct', 11: 'Nov', 12: 'Dec'}

print(f'  Input folder  : {INPUT_FOLDER}')
print(f'  Output folder : {OUTPUT_FOLDER}')
print(f'  Report roster : {len(REPORT_PERSONS)} people (hardcoded) — '
      + ', '.join(disp for _, disp in REPORT_PERSONS))
print(f'  Mouser guess  : {MOUSER_TARGET_LAKH_DEFAULT} Lakhs default, '
      f'{len(MOUSER_TARGET_LAKH_OVERRIDES)} override(s) on file')

  Input folder  : input_files
  Output folder : output_files
  Report roster : 6 people (hardcoded) — Kuldeep Kumar, Arpit Singh, Mahesh Kumar, Harsh Doshi, Suraj Pandey, Abhishekh Chaudhry
  Mouser guess  : 13 Lakhs default, 2 override(s) on file


### Step 2: Helper functions

In [5]:
def _clean(s):
    """Strip, collapse whitespace, uppercase, drop accents — comparison-ready but not yet
    matched to a canonical person (that happens in Step 3b)."""
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode('ascii')
    return ' '.join(s.strip().upper().split())


def _norm(s):
    """lowercase, collapse whitespace — for forgiving column/sheet-name matching"""
    return ' '.join(str(s).strip().lower().split())


def sheet_map(path):
    """{normalised sheet name: real sheet name} — handles trailing spaces like 'INR Sheet '"""
    return {_norm(s): s for s in pd.ExcelFile(path).sheet_names}


def load_sheet(path, *name_starts):
    """Read the first sheet whose normalised name starts with any of name_starts."""
    sm = sheet_map(path)
    real = None
    for want in name_starts:
        real = next((v for k, v in sm.items() if k.startswith(_norm(want))), None)
        if real:
            break
    if real is None:
        return None
    df = pd.read_excel(path, sheet_name=real, header=HEADER_ROW)
    df.columns = [str(c).strip() for c in df.columns]
    return df


def col_like(df, *substrings, exclude=()):
    """First column whose normalised name contains ALL substrings and none of `exclude`."""
    if df is None:
        return None
    for c in df.columns:
        cl = _norm(c)
        if all(s in cl for s in substrings) and not any(x in cl for x in exclude):
            return c
    return None


def find_month_val_col(columns, month_prefix):
    """The Base File's value column for one month, e.g. 'Aug' -> 'Aug_FY2627_Val'."""
    pat = re.compile(rf'^{re.escape(month_prefix)}.*_val$', re.IGNORECASE)
    for c in columns:
        if pat.match(str(c).strip()):
            return c
    return None


def sum_by(df, group_col, value_col, divisor, name_map):
    """Group by canonical salesman name, sum the value column, convert to reporting units."""
    if df is None or df.empty or not group_col or not value_col:
        return {}
    d = df[[group_col, value_col]].copy()
    d[group_col] = d[group_col].map(lambda x: name_map.get(_clean(x), _clean(x)))
    d = d[~d[group_col].isin(['', 'NAN', 'NONE'])]
    d[value_col] = pd.to_numeric(d[value_col], errors='coerce')
    g = d.groupby(group_col)[value_col].sum() / divisor
    return {k: float(v) for k, v in g.items() if pd.notna(v)}


def pct(ach, tgt):
    """Achievement / Target. None when there is no target to divide by."""
    if tgt is None or pd.isna(tgt) or tgt == 0:
        return None
    return ach / tgt


def fmt(v, nd=2):
    return '—' if v is None or pd.isna(v) else f'{v:,.{nd}f}'


def tint_color(rgb_hex, tint):
    """Excel's theme-tint algorithm (HLS lightening), so a hardcoded RGB fill looks exactly
    like the theme-coloured fill in the reference file, without depending on workbook theme."""
    r, g, b = (int(rgb_hex[i:i + 2], 16) / 255 for i in (0, 2, 4))
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = l * (1.0 + tint) if tint < 0 else l * (1.0 - tint) + tint
    r2, g2, b2 = colorsys.hls_to_rgb(h, l, s)
    return ''.join('%02X' % int(round(min(max(x, 0), 1) * 255)) for x in (r2, g2, b2))


print('✓ Helpers ready')

✓ Helpers ready


### Step 3: Discover the input files

The Base File is identified by its **columns** (`Currency` + `Sales Incharge`), not by sheet name, so it can live on any single sheet under any name.

In [6]:
folder = Path(INPUT_FOLDER)
if not folder.exists():
    raise SystemExit(f'Folder "{INPUT_FOLDER}" does not exist. Create it and add the source files.')

all_files = sorted(f for f in list(folder.glob('*.xlsx')) + list(folder.glob('*.xls'))
                   if not f.name.startswith('~$'))


def is_base_file(path):
    """A Base/targets file: some sheet has both a 'Currency' and a 'Sales Incharge' column."""
    for real in sheet_map(path).values():
        try:
            d = pd.read_excel(path, sheet_name=real, header=HEADER_ROW, nrows=5)
        except Exception:
            continue
        cn = [_norm(c) for c in d.columns]
        if 'currency' in cn and any('sales incharge' in c for c in cn):
            return True
    return False


base_files, billing_files = [], []
for f in all_files:
    if is_base_file(f):
        base_files.append(f)
    else:
        sm = sheet_map(f)
        if 'line' in sm or 'mouser' in sm:
            billing_files.append(f)


def billing_last_date(path):
    """Latest real invoice/posting date — used to pick the newest export if several exist."""
    best = None
    sm = sheet_map(path)
    for key in ('line', 'mouser', 'usd', 'direct'):
        real = sm.get(key)
        if not real:
            continue
        d = pd.read_excel(path, sheet_name=real, header=HEADER_ROW)
        d.columns = [str(c).strip() for c in d.columns]
        for c in d.columns:
            cl = _norm(c)
            if 'po date' in cl or 'order date' in cl:
                continue                       # customer PO date is not a billing date
            if any(t in cl for t in ('inv date', 'posting date', 'document date')):
                dts = pd.to_datetime(d[c], errors='coerce').dropna()
                dts = dts[dts.dt.year >= 2000]
                if len(dts):
                    best = dts.max() if best is None else max(best, dts.max())
    return best


print('FILE DISCOVERY')
print('=' * 74)
print(f'  Files in {INPUT_FOLDER}: {len(all_files)}')

if not base_files:
    raise SystemExit('No Base File found (expected a sheet with "Currency" + "Sales Incharge" columns).')
if not billing_files:
    raise SystemExit('No Billing Report found (expected a "Line" / "MOUSER" sheet).')

BASE_FILE = base_files[0]
print(f'\n  \U0001F4D8 Base File (targets)  : {BASE_FILE.name}')
if len(base_files) > 1:
    print(f'      \u26A0\uFE0F  {len(base_files)} base files present; using the first. '
          f'Others: {", ".join(f.name for f in base_files[1:])}')

dated = sorted(((f, billing_last_date(f)) for f in billing_files),
               key=lambda t: (t[1] is not None, t[1]))
BILLING_FILE, BILLING_UPTO = dated[-1]

print(f'\n  \U0001F4D7 Billing Reports      : {len(billing_files)}')
for f, d in dated:
    upto = d.strftime('%Y-%m-%d') if d is not None else 'no dates'
    tag = '  <- USING (newest)' if f is BILLING_FILE else '     skipped (older export)'
    print(f'       {f.name:44} up to {upto}{tag}')

if BILLING_UPTO is None:
    raise SystemExit('Could not find any invoice date in the Billing Report — cannot '
                      'determine the reporting month.')

MONTH_LABEL = BILLING_UPTO.strftime("%b'%y")
MONTH_PFX = MONTH_PREFIX.get(BILLING_UPTO.month)
print(f"\n  \U0001F4C5 Reporting month detected: {MONTH_LABEL}  (Base File column prefix: {MONTH_PFX})")

FILE DISCOVERY
  Files in input_files: 2

  📘 Base File (targets)  : targets.xlsx

  📗 Billing Reports      : 1
       Aug_Billing_Report.xlsx                      up to 2026-08-21  <- USING (newest)

  📅 Reporting month detected: Aug'26  (Base File column prefix: Aug)


### Step 3b: Build the salesperson name list (automatic, fuzzy-matched)

Instead of a hand-maintained alias table, every raw salesperson name found in either file is compared against every other with a string-similarity score. Names that score above `NAME_MATCH_THRESHOLD` are treated as the same person — this is what catches `DHIRENDRA PANDEY` vs `DHIRENDER PANDEY` without anyone having to know in advance that the typo exists. Every merge is printed below for a human to sanity-check.

In [7]:
def collect_raw_names():
    """Every raw value seen in a salesperson column, across both input files."""
    names = []

    base_df_peek = None
    for real in sheet_map(BASE_FILE).values():
        d = pd.read_excel(BASE_FILE, sheet_name=real, header=HEADER_ROW)
        d.columns = [str(c).strip() for c in d.columns]
        cn = [_norm(c) for c in d.columns]
        if 'currency' in cn and any('sales incharge' in c for c in cn):
            base_df_peek = d
            break
    if base_df_peek is not None:
        sp_col = col_like(base_df_peek, 'sales', 'incharge')
        if sp_col:
            names += base_df_peek[sp_col].dropna().tolist()

    for sheet, sp_parts in (('line', ('sales', 'employee')), ('mouser', ('sales', 'employee')),
                             ('usd', ('sales', 'employee')), ('direct', ('sales', 'person'))):
        df = load_sheet(BILLING_FILE, sheet)
        sp_col = col_like(df, *sp_parts)
        if sp_col:
            names += df[sp_col].dropna().tolist()

    return [n for n in names if str(n).strip()]


def build_name_map(raw_names, threshold):
    """Cluster near-duplicate names; canonical form = the longest spelling in the cluster
    (ties broken alphabetically), which in practice is almost always the fuller/correct one."""
    uniq = sorted({_clean(n) for n in raw_names})
    clusters = []
    for n in uniq:
        home = None
        for cluster in clusters:
            if any(SequenceMatcher(None, n, m).ratio() >= threshold for m in cluster):
                home = cluster
                break
        if home is None:
            clusters.append([n])
        else:
            home.append(n)

    name_map = {}
    merges = []
    for cluster in clusters:
        canon = sorted(cluster, key=lambda s: (-len(s), s))[0]
        for n in cluster:
            name_map[n] = canon
            if n != canon:
                merges.append((n, canon))

    for raw, canon in MANUAL_NAME_OVERRIDES.items():
        name_map[_clean(raw)] = _clean(canon)

    return name_map, merges


raw_names = collect_raw_names()
NAME_MAP, merges = build_name_map(raw_names, NAME_MATCH_THRESHOLD)

print('SALESPERSON NAME MATCHING')
print('=' * 74)
print(f'  Raw name variants seen : {len(set(_clean(n) for n in raw_names))}')
print(f'  Resolved to            : {len(set(NAME_MAP.values()))} distinct people')
if merges:
    print(f'\n  Auto-merged as the same person (similarity >= {NAME_MATCH_THRESHOLD:.0%}):')
    for raw, canon in merges:
        print(f'    "{raw}"  ->  "{canon}"')
else:
    print('\n  No near-duplicate names found this run.')
if MANUAL_NAME_OVERRIDES:
    print(f'\n  Manual overrides applied: {len(MANUAL_NAME_OVERRIDES)}')

report_keys = {k for k, _ in REPORT_PERSONS}
missing = report_keys - set(NAME_MAP.values())
if missing:
    print(f'\n  \u26A0\uFE0F  REPORT_PERSONS has {len(missing)} name(s) not found in either input '
          f'file at all: {missing}')
    print('     They will show up with target/achievement = 0 below — check the spelling in')
    print('     REPORT_PERSONS (Step 1) against the names printed above.')

print('\n  \u26A0\uFE0F  Skim the merge list above — if two DIFFERENT people got merged, lower '
      'NAME_MATCH_THRESHOLD or add them to MANUAL_NAME_OVERRIDES in Step 1.')

SALESPERSON NAME MATCHING
  Raw name variants seen : 11
  Resolved to            : 10 distinct people

  Auto-merged as the same person (similarity >= 90%):
    "DHIRENDRA PANDEY"  ->  "DHIRENDER PANDEY"

  ⚠️  Skim the merge list above — if two DIFFERENT people got merged, lower NAME_MATCH_THRESHOLD or add them to MANUAL_NAME_OVERRIDES in Step 1.


### Step 4: TARGETS from the Base File

In [8]:
targets = {'inr': {}, 'usdsg': {}, 'usdd': {}, 'mou': {}, 'jns': {}}

print('TARGETS  (source: %s)' % BASE_FILE.name)
print('=' * 74)

base_df = None
for real in sheet_map(BASE_FILE).values():
    d = pd.read_excel(BASE_FILE, sheet_name=real, header=HEADER_ROW)
    d.columns = [str(c).strip() for c in d.columns]
    cn = [_norm(c) for c in d.columns]
    if 'currency' in cn and any('sales incharge' in c for c in cn):
        base_df = d
        break

VAL_COL  = find_month_val_col(base_df.columns, MONTH_PFX) if base_df is not None else None
CUR_COL  = col_like(base_df, 'currency')
SP_COL   = col_like(base_df, 'sales', 'incharge')
CUST_COL = col_like(base_df, 'customer', 'name')

if base_df is None or VAL_COL is None:
    raise SystemExit(f'Could not find a target value column for "{MONTH_PFX}" in the Base '
                      f'File — refusing to publish a report with zero targets.')

cur = base_df[CUR_COL].astype(str).str.strip().str.upper()

sub_inr = base_df[cur == 'INR']
targets['inr'] = sum_by(sub_inr, SP_COL, VAL_COL, CR, NAME_MAP)
print(f'  \u2713 INR    (Line business) -> "{VAL_COL}" by "{SP_COL}"  /{CR:,}'
      f'  = {sum(targets["inr"].values()):,.4f} Cr')

sub_usd = base_df[cur == 'USD']
targets['usdsg'] = sum_by(sub_usd, SP_COL, VAL_COL, MILL, NAME_MAP)
print(f'  \u2713 USD    (USD-SG business) -> "{VAL_COL}" by "{SP_COL}"  /{MILL:,}'
      f'  = {sum(targets["usdsg"].values()):,.4f} M$')

sub_dir = base_df[cur == 'DIRECT']
targets['usdd'] = sum_by(sub_dir, SP_COL, VAL_COL, MILL, NAME_MAP)
print(f'  \u2713 DIRECT (USD-Direct business) -> "{VAL_COL}" by "{SP_COL}"  /{MILL:,}'
      f'  = {sum(targets["usdd"].values()):,.4f} M$')

if INCLUDE_JNS_TFT and CUST_COL:
    is_jns = sub_usd[CUST_COL].astype(str).str.upper().str.contains(JNS_CUSTOMER_MATCH, na=False)
    targets['jns'] = sum_by(sub_usd[is_jns], SP_COL, VAL_COL, MILL, NAME_MAP)
    if targets['jns']:
        print(f'  \u2713 JNS (TFT) target (USD rows where Customer Name contains '
              f'"{JNS_CUSTOMER_MATCH}") = {sum(targets["jns"].values()):,.4f} M$'
              f'  -- an ADD-ON figure, already counted once inside USD-SG above')

base_roster = set(targets['inr']) | set(targets['usdsg']) | set(targets['usdd']) | set(targets['jns'])
print(f'\n  \U0001F464 Salesmen in Base File   : {len(base_roster)}')

TARGETS  (source: targets.xlsx)
  ✓ INR    (Line business) -> "Aug_FY2627_Val" by "Sales Incharge"  /10,000,000  = 17.9729 Cr
  ✓ USD    (USD-SG business) -> "Aug_FY2627_Val" by "Sales Incharge"  /1,000,000  = 6.1911 M$
  ✓ DIRECT (USD-Direct business) -> "Aug_FY2627_Val" by "Sales Incharge"  /1,000,000  = 0.3602 M$
  ✓ JNS (TFT) target (USD rows where Customer Name contains "JNS") = 1.2677 M$  -- an ADD-ON figure, already counted once inside USD-SG above

  👤 Salesmen in Base File   : 9


### Step 5: ACHIEVEMENT from the Billing Report

Achievement is summed from **Basic Value** (pre-tax/pre-freight), not 'Total Value' — the Billing Report's own printed grand-total row (Line 1 of each sheet) confirms Basic Value is the figure that adds up to the numbers actually reported.

In [9]:
ach = {'inr': {}, 'usdsg': {}, 'usdd': {}, 'mou': {}, 'jns': {}}

print('ACHIEVEMENT  (source: %s)' % BILLING_FILE.name)
print('=' * 74)

for key, sheet, divisor, unit in (('inr', 'line', CR, 'Cr'), ('mou', 'mouser', LAKH, 'Lakhs')):
    df = load_sheet(BILLING_FILE, sheet)
    if df is None or df.empty:
        print(f'  \u2022  {sheet.upper():7} sheet empty or missing -- contributes 0')
        continue
    val_col = col_like(df, 'basic', 'value')
    sp_col  = col_like(df, 'sales', 'employee')
    if not val_col or not sp_col:
        print(f'  \u26A0\uFE0F  {sheet.upper():7} value or salesman column not found -- skipped')
        continue
    ach[key] = sum_by(df, sp_col, val_col, divisor, NAME_MAP)
    print(f'  \u2713 {sheet.upper():7} -> "{val_col}" by "{sp_col}"  /{divisor:,}'
          f'  = {sum(ach[key].values()):,.4f} {unit}  ({len(df)} rows)')

df = load_sheet(BILLING_FILE, 'usd')
if df is not None and not df.empty:
    val_col  = col_like(df, 'row total', 'usd')
    sp_col   = col_like(df, 'sales', 'employee')
    cust_col = col_like(df, 'customer', 'name')
    if INCLUDE_JNS_TFT and cust_col:
        is_jns = df[cust_col].astype(str).str.upper().str.contains(JNS_CUSTOMER_MATCH, na=False)
        ach['usdsg'] = sum_by(df[~is_jns], sp_col, val_col, MILL, NAME_MAP)
        ach['jns']   = sum_by(df[is_jns],  sp_col, val_col, MILL, NAME_MAP)
        print(f'  \u2713 USD     -> "{val_col}" by "{sp_col}"  /{MILL:,}'
              f'  = {sum(ach["usdsg"].values()):,.4f} M$  ({(~is_jns).sum()} rows,'
              f' {is_jns.sum()} rows carved out to JNS (TFT) = {sum(ach["jns"].values()):,.4f} M$)')
    else:
        ach['usdsg'] = sum_by(df, sp_col, val_col, MILL, NAME_MAP)
        print(f'  \u2713 USD     -> "{val_col}" by "{sp_col}"  /{MILL:,}'
              f'  = {sum(ach["usdsg"].values()):,.4f} M$  ({len(df)} rows)')
else:
    print('  \u2022  USD     sheet empty or missing -- contributes 0')

df = load_sheet(BILLING_FILE, 'direct')
if df is not None and not df.empty:
    val_col = col_like(df, 'row total', 'usd')
    sp_col  = col_like(df, 'sales', 'person')
    ach['usdd'] = sum_by(df, sp_col, val_col, MILL, NAME_MAP)
    print(f'  \u2713 DIRECT  -> "{val_col}" by "{sp_col}"  /{MILL:,}'
          f'  = {sum(ach["usdd"].values()):,.4f} M$  ({len(df)} rows)')
else:
    print('  \u2022  DIRECT  sheet empty or missing -- contributes 0')

billed = set().union(*(set(v) for v in ach.values())) if any(ach.values()) else set()
print(f'\n  \U0001F464 Salesmen with billing   : {len(billed)}')

ACHIEVEMENT  (source: Aug_Billing_Report.xlsx)
  ✓ LINE    -> "Basic Value" by "Sales Employee Name"  /10,000,000  = 5.2124 Cr  (300 rows)
  ✓ MOUSER  -> "Basic Value" by "Sales Employee Name"  /100,000  = 69.5629 Lakhs  (654 rows)
  ✓ USD     -> "Row Total in USD" by "Sales Employee"  /1,000,000  = 0.9170 M$  (177 rows, 4 rows carved out to JNS (TFT) = 0.4306 M$)
  ✓ DIRECT  -> "Row Total in USD" by "Sales Person"  /1,000,000  = 0.0311 M$  (13 rows)

  👤 Salesmen with billing   : 10


### Step 6: Mouser targets (hand-entered — see Step 1)

The Base File has zero Mouser rows, so there is no real target to read. Everyone in the report gets the flat guess from `MOUSER_TARGET_LAKH_DEFAULT` unless a confirmed number is in `MOUSER_TARGET_LAKH_OVERRIDES`.

In [10]:
for k, _ in REPORT_PERSONS:
    targets['mou'][k] = MOUSER_TARGET_LAKH_OVERRIDES.get(k, MOUSER_TARGET_LAKH_DEFAULT)

print('MOUSER TARGETS')
print('=' * 74)
confirmed = [k for k, _ in REPORT_PERSONS if k in MOUSER_TARGET_LAKH_OVERRIDES]
guessed = [k for k, _ in REPORT_PERSONS if k not in MOUSER_TARGET_LAKH_OVERRIDES]
print(f'  {len(confirmed)} confirmed : ' +
      (", ".join(f"{n}={MOUSER_TARGET_LAKH_OVERRIDES[n]}L" for n in confirmed) or "none"))
print(f'  {len(guessed)} on the flat {MOUSER_TARGET_LAKH_DEFAULT}L guess : ' +
      (", ".join(guessed) or "none"))
print('\n  This is the one hand-entered figure in the whole report — update Step 1 as soon')
print('  as more real per-person Mouser targets are confirmed.')

MOUSER TARGETS
  2 confirmed : HARSH DOSHI=55L, ABHISHEKH CHOUDHARY=3L
  4 on the flat 13L guess : KULDEEP KUMAR, ARPIT SINGH, MAHESH KUMAR MITTAL, SURAJ PANDEY

  This is the one hand-entered figure in the whole report — update Step 1 as soon
  as more real per-person Mouser targets are confirmed.


### Step 7: Assemble the report

Matching the reference file's own design: the **Company Summary is company-wide** — every salesperson found in the input files, whether or not they're named below — and only the **per-person breakdown** is restricted to `REPORT_PERSONS`. JNS-TFT is a summary-only line in the reference file (no per-person JNS row), so it's handled the same way here.

In [11]:
PERSON_ROWS = [('INR (Cr)', 'inr'), ('USD-SG (M$)', 'usdsg'),
               ('USD-Direct (M$)', 'usdd'), ('Mouser (Lakhs)', 'mou')]

company_roster = sorted(base_roster | billed)          # everyone, for the summary block
display_roster = [k for k, _ in REPORT_PERSONS]         # hardcoded subset, for the breakdown
display_name = dict(REPORT_PERSONS)

table = {n: {k: (targets[k].get(n), ach[k].get(n, 0.0), pct(ach[k].get(n, 0.0),
                                                            targets[k].get(n)))
             for _, k in PERSON_ROWS} for n in display_roster}


def sum_t(k, roster, scale=1.0):
    """Total target across `roster`. None (not 0) when nobody in it has a target."""
    vals = [targets[k].get(n) for n in roster]
    vals = [v for v in vals if v is not None]
    return sum(vals) / scale if vals else None


def sum_a(k, roster, scale=1.0):
    return sum(ach[k].get(n, 0.0) for n in roster) / scale


summary = [('INR (Cr)',        sum_t('inr', company_roster),        sum_a('inr', company_roster),        2),
           ('USD-SG (M$)',     sum_t('usdsg', company_roster),      sum_a('usdsg', company_roster),      2),
           ('USD-Direct (M$)', sum_t('usdd', company_roster),       sum_a('usdd', company_roster),       3),
           # Mouser targets only exist for REPORT_PERSONS (Step 6) even though the summary
           # roster is company-wide — sum_t simply skips anyone with no target set, so this
           # naturally becomes "target = the 6 confirmed/guessed reps, achievement = everyone".
           ('Mouser (Cr)',     sum_t('mou', company_roster, 100),   sum_a('mou', company_roster, 100),   2)]
if INCLUDE_JNS_TFT and (targets['jns'] or ach['jns']):
    summary.append(('JNS-TFT (M$)', sum_t('jns', company_roster), sum_a('jns', company_roster), 3))

print(f"Billing for {MONTH_LABEL}   (company roster: {len(company_roster)}, "
      f"report roster: {len(display_roster)})")
print('=' * 74)
print(f'  {"":18} {"TGT":>10} {"ACT":>10} {"%":>7}')
for label, t, a, nd in summary:
    p = pct(a, t)
    print(f'  {label:18} {fmt(t, nd):>10} {fmt(a, nd):>10} '
          f'{(f"{p:.0%}" if p is not None else "-"):>7}')

print(f'\n{"=" * 74}\nBY SALESMAN (report roster only)\n{"=" * 74}')
for n in display_roster:
    print(f'\n  {display_name[n]}')
    for label, key in PERSON_ROWS:
        t, a, p = table[n][key]
        nd = 3 if key == 'usdd' else 2
        print(f'     {label:16} Target {fmt(t, nd):>9}   Ach. {fmt(a, nd):>9}   '
              f'{(f"{p:.0%}" if p is not None else "-"):>6}')

Billing for Aug'26   (company roster: 10, report roster: 6)
                            TGT        ACT       %
  INR (Cr)                17.97       5.21     29%
  USD-SG (M$)              6.19       0.92     15%
  USD-Direct (M$)         0.360      0.031      9%
  Mouser (Cr)              1.10       0.70     63%
  JNS-TFT (M$)            1.268      0.431     34%

BY SALESMAN (report roster only)

  Kuldeep Kumar
     INR (Cr)         Target      3.58   Ach.      1.39      39%
     USD-SG (M$)      Target      2.32   Ach.      0.58      25%
     USD-Direct (M$)  Target     0.107   Ach.     0.015      14%
     Mouser (Lakhs)   Target     13.00   Ach.      2.98      23%

  Arpit Singh
     INR (Cr)         Target      5.24   Ach.      0.74      14%
     USD-SG (M$)      Target      1.18   Ach.      0.19      16%
     USD-Direct (M$)  Target         —   Ach.     0.000        -
     Mouser (Lakhs)   Target     13.00   Ach.      6.97      54%

  Mahesh Kumar
     INR (Cr)         Target    

### Step 8: Validation — sanity checks before anything gets published

This is the step that matters most for a CTO/CXO-facing report. It re-derives each category's *company-wide* total two different ways (once as the raw sheet total, once as the sum across everyone found in the files) so a name-matching problem shows up here even if it doesn't affect anyone on the `REPORT_PERSONS` list. It also flags anything the report itself should have a human look at.

In [12]:
problems = []


def check_total(label, achievement_dict, raw_total):
    computed = sum(achievement_dict.values())
    if raw_total is None:
        return
    diff = computed - raw_total
    status = 'OK' if abs(diff) < 0.01 else 'MISMATCH'
    print(f'  [{status:8}] {label:26} everyone-in-files sum = {computed:,.4f}   '
          f'sheet total = {raw_total:,.4f}   diff = {diff:,.4f}')
    if status == 'MISMATCH':
        problems.append(f'{label}: total achievement ({computed:,.4f}) does not match the '
                         f'raw sheet total ({raw_total:,.4f}) — a name is probably not '
                         f'matching correctly (see Step 3b).')


def sheet_total(sheet, col_parts):
    df = load_sheet(BILLING_FILE, sheet)
    col = col_like(df, *col_parts)
    return pd.to_numeric(df[col], errors='coerce').sum() if (df is not None and col) else None


print('RECONCILIATION — company-wide achievement vs. raw sheet totals')
print('=' * 74)
check_total('INR achievement',    ach['inr'],   sheet_total('line', ('basic', 'value')) / CR)
check_total('Mouser achievement', ach['mou'],   sheet_total('mouser', ('basic', 'value')) / LAKH)
usd_raw = sheet_total('usd', ('row total', 'usd'))
check_total('USD-SG + JNS achievement',
            {**ach['usdsg'], '__jns__': sum(ach['jns'].values())},
            usd_raw / MILL if usd_raw is not None else None)
check_total('USD-Direct achievement', ach['usdd'], sheet_total('direct', ('row total', 'usd')) / MILL)

print(f'\n{"=" * 74}\nREPORT-ROSTER CHECKS\n{"=" * 74}')

excluded_with_billing = billed - set(display_roster)
if excluded_with_billing:
    excl_total = sum(ach['inr'].get(n, 0) for n in excluded_with_billing)
    print(f'  \u2139\uFE0F  {len(excluded_with_billing)} salesperson(s) have real billing this month '
          f'but are not in REPORT_PERSONS (they still count in the Company Summary above): '
          f'{sorted(excluded_with_billing)}')
    print(f'     (their combined INR achievement is {excl_total:,.4f} Cr)')

for n in display_roster:
    if targets['inr'].get(n) is None and ach['inr'].get(n):
        problems.append(f'{display_name[n]}: has INR billing but no INR target in the Base File.')
    for label, key in PERSON_ROWS:
        p = table[n][key][2]
        if p is not None and p > 3.0:
            problems.append(f'{display_name[n]}: {label} achievement is {p:.0%} of target — '
                             f'check whether this billing is really theirs (see Step 3b/5).')

if problems:
    print(f'\n  \u26A0\uFE0F  {len(problems)} item(s) need a human look before this goes out:')
    for p in problems:
        print(f'     - {p}')
else:
    print('\n  \u2713 Nothing flagged.')

print(f'\n{"=" * 74}')
if problems:
    print(f'  Proceeding to save the report anyway — these are WARNINGS, not blockers — but')
    print(f'  read the list above before forwarding it to the CTO/CXO.')
else:
    print(f'  \u2713 All reconciliation checks passed.')

RECONCILIATION — company-wide achievement vs. raw sheet totals
  [OK      ] INR achievement            everyone-in-files sum = 5.2124   sheet total = 5.2124   diff = 0.0000
  [OK      ] Mouser achievement         everyone-in-files sum = 69.5629   sheet total = 69.5629   diff = 0.0000
  [OK      ] USD-SG + JNS achievement   everyone-in-files sum = 1.3476   sheet total = 1.3476   diff = 0.0000
  [OK      ] USD-Direct achievement     everyone-in-files sum = 0.0311   sheet total = 0.0311   diff = 0.0000

REPORT-ROSTER CHECKS
  ℹ️  4 salesperson(s) have real billing this month but are not in REPORT_PERSONS (they still count in the Company Summary above): ['DHIRENDER PANDEY', 'RAJIV ANAND', 'RISHABH PANDEY', 'SANDHYA KALYAN']
     (their combined INR achievement is 0.1602 Cr)

  ✓ Nothing flagged.

  ✓ All reconciliation checks passed.


### Step 9: Write the Excel file — styled to match `Output.xlsx` exactly

In [13]:
# Colours lifted directly from the reference Output.xlsx:
SUMMARY_FILL = PatternFill('solid', fgColor='FF94DCF8')                 # summary band (plain RGB in the sample)
HEADER_FILL  = PatternFill('solid', fgColor='FF' + tint_color('5B9BD5', 0.7499))  # per-person header (theme colour in the sample, reproduced from its RGB + tint)
NAVY   = 'FF002060'
BOLD          = Font(bold=True)
NAVY_BOLD     = Font(bold=True, color=NAVY)
NAVY_REGULAR  = Font(color=NAVY)
CTR  = Alignment(horizontal='center', vertical='center', wrap_text=True)
CTR_NOWRAP = Alignment(horizontal='center', vertical='center')
THIN   = Side(style='thin')
MEDIUM = Side(style='medium')
DASH = '\u2014'

wb = openpyxl.Workbook()
ws = wb.active
ws.title = 'Sheet1'


def put(r, c, value, numfmt=None, font=None, fill=None, align=None, border=None):
    cell = ws.cell(row=r, column=c, value=value)
    if numfmt: cell.number_format = numfmt
    if font:   cell.font = font
    if fill:   cell.fill = fill
    if align:  cell.alignment = align
    if border: cell.border = border
    return cell


def thin_box():
    return Border(left=THIN, right=THIN, top=THIN, bottom=THIN)


# ══ SUMMARY BLOCK (rows 2-8, matches the sample's A2:D8 layout) ═══════════
ws.merge_cells('A2:D2')
put(2, 1, f"Billing for {MONTH_LABEL}", font=BOLD, fill=SUMMARY_FILL, align=CTR, border=thin_box())
for c in range(2, 5):
    put(2, c, None, fill=SUMMARY_FILL, border=thin_box())
ws.row_dimensions[2].height = 16.4

put(3, 1, None, fill=SUMMARY_FILL, border=thin_box())
for c, h in enumerate(('TGT', 'ACT', '%'), start=2):
    put(3, c, h, font=BOLD, fill=SUMMARY_FILL, align=CTR_NOWRAP, border=thin_box())

for i, (label, t, a, nd) in enumerate(summary):
    r = 4 + i
    numfmt = '0.000' if nd == 3 else '0.00'
    put(r, 1, label, border=thin_box())
    if t is None:
        put(r, 2, DASH, align=CTR_NOWRAP, border=thin_box())
    else:
        put(r, 2, round(t, 6), numfmt=numfmt, border=thin_box())
    put(r, 3, round(a, 6), numfmt=numfmt, border=thin_box())
    if t:
        put(r, 4, f'=C{r}/B{r}', numfmt='0%', border=thin_box())
    else:
        put(r, 4, DASH, align=CTR_NOWRAP, border=thin_box())

# ══ PER-SALESMAN BLOCK (matches the sample's row 11 header / row 13+ data) ═
HDR = 11
TOP = HDR + 2

ws.merge_cells(start_row=HDR, start_column=1, end_row=HDR + 1, end_column=1)
put(HDR, 1, 'Date', font=NAVY_BOLD, fill=HEADER_FILL, align=CTR_NOWRAP, border=Border(left=MEDIUM, right=MEDIUM, top=MEDIUM, bottom=MEDIUM))
put(HDR + 1, 1, None, fill=HEADER_FILL, border=Border(left=MEDIUM, right=MEDIUM, bottom=MEDIUM))
ws.merge_cells(start_row=HDR, start_column=2, end_row=HDR + 1, end_column=2)
put(HDR, 2, 'Particulars', font=NAVY_BOLD, fill=HEADER_FILL, align=CTR_NOWRAP, border=Border(left=MEDIUM, right=MEDIUM, top=MEDIUM, bottom=MEDIUM))
put(HDR + 1, 2, None, fill=HEADER_FILL, border=Border(left=MEDIUM, right=MEDIUM, bottom=MEDIUM))

for i, name in enumerate(display_roster):
    c0 = 3 + i * 4                                    # C, G, K, O, ...
    ws.merge_cells(start_row=HDR, start_column=c0, end_row=HDR, end_column=c0 + 2)
    put(HDR, c0, display_name[name], font=NAVY_BOLD, fill=HEADER_FILL, align=CTR_NOWRAP,
        border=Border(left=MEDIUM, right=MEDIUM, top=MEDIUM, bottom=MEDIUM))
    for off in (1, 2):
        put(HDR, c0 + off, None, fill=HEADER_FILL,
            border=Border(top=MEDIUM, bottom=MEDIUM, right=(MEDIUM if off == 2 else None)))
    for j, h in enumerate(('Target', 'Ach.', '%')):
        put(HDR + 1, c0 + j, h, font=NAVY_BOLD, fill=HEADER_FILL, align=CTR_NOWRAP,
            border=Border(right=MEDIUM, bottom=MEDIUM))

n_rows = len(PERSON_ROWS)
ws.merge_cells(start_row=TOP, start_column=1, end_row=TOP + n_rows - 1, end_column=1)
put(TOP, 1, MONTH_LABEL, font=NAVY_REGULAR, align=CTR,
    border=Border(left=MEDIUM, right=MEDIUM, top=MEDIUM, bottom=MEDIUM))
for r in range(TOP + 1, TOP + n_rows):
    put(r, 1, None, border=Border(left=MEDIUM, right=MEDIUM))

for ri, (label, key) in enumerate(PERSON_ROWS):
    r = TOP + ri
    numfmt = '0.000' if key == 'usdd' else '0.00'
    is_first, is_last = (ri == 0), (ri == n_rows - 1)
    put(r, 2, label, font=NAVY_REGULAR,
        border=Border(right=MEDIUM, top=(MEDIUM if is_first else None), bottom=MEDIUM))
    for i, name in enumerate(display_roster):
        t, a, _ = table[name][key]
        c = 3 + i * 4
        top = MEDIUM if is_first else None
        tgt_fmt = 'General' if key == 'mou' else numfmt
        if t is None:
            put(r, c, DASH, font=NAVY_REGULAR, align=CTR_NOWRAP,
                border=Border(right=MEDIUM, top=top, bottom=MEDIUM))
        else:
            put(r, c, round(t, 6), numfmt=tgt_fmt, font=NAVY_REGULAR,
                border=Border(right=MEDIUM, top=top, bottom=MEDIUM))
        put(r, c + 1, round(a, 6), numfmt=numfmt, font=NAVY_REGULAR,
            border=Border(right=MEDIUM, top=top, bottom=MEDIUM))
        if t:
            put(r, c + 2, f'={get_column_letter(c+1)}{r}/{get_column_letter(c)}{r}', numfmt='0%',
                font=NAVY_REGULAR, border=Border(right=MEDIUM, top=top, bottom=MEDIUM))
        else:
            put(r, c + 2, DASH, font=NAVY_REGULAR, align=CTR_NOWRAP,
                border=Border(right=MEDIUM, top=top, bottom=MEDIUM))

# ══ COLUMN WIDTHS (spacer column between each person, as in the sample) ═══
ws.column_dimensions['A'].width = 14.37
ws.column_dimensions['B'].width = 13.0
for i in range(len(display_roster)):
    c0 = 3 + i * 4
    for off, w in ((0, 7.5), (1, 7.5), (2, 6.5), (3, 2.0)):
        ws.column_dimensions[get_column_letter(c0 + off)].width = w

out_dir = Path(OUTPUT_FOLDER)
out_dir.mkdir(parents=True, exist_ok=True)
timestamp = RUN_STARTED.strftime('%Y%m%d_%H%M%S')
month_tag = BILLING_UPTO.strftime('%b%y')
OUTPUT_FILE = out_dir / f'Billing_Summary_{month_tag}_{timestamp}.xlsx'
wb.save(OUTPUT_FILE)

print('=' * 74)
print(f'\u2705 Saved: {OUTPUT_FILE}')
print('=' * 74)
print(f'   Month         : {MONTH_LABEL}   (detected from the Billing Report dates)')
print(f'   Targets from  : {BASE_FILE.name}')
print(f'   Billing from  : {BILLING_FILE.name}'
      + (f'   (up to {BILLING_UPTO:%Y-%m-%d})' if BILLING_UPTO is not None else ''))
print(f'   Salesmen      : {len(display_roster)}  (hardcoded roster — see REPORT_PERSONS in Step 1)')
print(f'   Summary rows  : {len(summary)}')
print(f'   Warnings      : {len(problems)}')
if problems:
    print(f'   \u26A0\uFE0F  Re-read Step 8 before sending this to the CTO/CXO.')

✅ Saved: output_files/Billing_Summary_Aug26_20260823_074251.xlsx
   Month         : Aug'26   (detected from the Billing Report dates)
   Targets from  : targets.xlsx
   Billing from  : Aug_Billing_Report.xlsx   (up to 2026-08-21)
   Salesmen      : 6  (hardcoded roster — see REPORT_PERSONS in Step 1)
   Summary rows  : 5
   Warnings      : 0
